In [9]:
#Import the necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import xgboost as xgb

In [16]:
#Read in the data and look at the first few rows
spotify_tracks = pd.read_csv('spotify_tracks.csv')
spotify_tracks.head()

,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,...,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,...,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,...,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,...,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,...,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


In [17]:
spotify_tracks = spotify_tracks[spotify_tracks['track_genre'].isin(['pop', 'country', 'hip-hop', 'punk-rock', 'latin', 'edm'])]

In [18]:
#Drop non-numeric and unnecessary columns and clean up missing values
spotify_tracks.drop(columns=["Unnamed: 0", "track_id", "track_name", "artists", "album_name", "time_signature"], axis=1, inplace=True)

# Check for missing values
print(spotify_tracks.isnull().sum())
spotify_tracks.dropna(inplace=True)

popularity          0
duration_ms         0
explicit            0
danceability        0
energy              0
key                 0
loudness            0
mode                0
speechiness         0
acousticness        0
instrumentalness    0
liveness            0
valence             0
tempo               0
track_genre         0
dtype: int64


In [19]:
#Define features and target and split dataset
X, y = spotify_tracks.loc[ : , (spotify_tracks.columns != 'track_genre')], spotify_tracks['track_genre']

#Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
#Split train further to train/validation
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=42)

In [ ]:
label_encoder = LabelEncoder()
y_train= label_encoder.fit_transform(y_train)
y_val = label_encoder.transform(y_val)
y_test = label_encoder.transform(y_test)    #turns it into numeric data

In [21]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)       # make sure all numeric values are on the same scale

In [23]:
xgb_classifier = xgb.XGBClassifier(random_state = 42)

xgb_classifier.fit(X_train, y_train)    #fit this model into x and y data

y_pred = xgb_classifier.predict(X_val)  #need to see how well this performs onto validation set

test_accuracy = accuracy_score(y_val, y_pred)       #yval = actual class, ypred = model predicted class
print(f"Test Accuracy : {test_accuracy:.4f}")

# Print classification report
print("\nClassification Report:\n", classification_report(y_val, y_pred))

# Confusion Matrix
print("\nConfusion Matrix:\n", confusion_matrix(y_val, y_pred))

Test Accuracy : 0.7083

Classification Report:
               precision    recall  f1-score   support

           0       0.73      0.83      0.78       197
           1       0.77      0.73      0.75       208
           2       0.63      0.60      0.61       206
           3       0.70      0.63      0.66       202
           4       0.57      0.62      0.59       185
           5       0.84      0.84      0.84       202

    accuracy                           0.71      1200
   macro avg       0.71      0.71      0.71      1200
weighted avg       0.71      0.71      0.71      1200


Confusion Matrix:
 [[164   3   3   6   9  12]
 [ 10 152  12   5  19  10]
 [  3  14 123  28  36   2]
 [ 18   7  25 127  20   5]
 [ 11  15  27  14 115   3]
 [ 18   7   4   1   3 169]]


In [ ]:
#Use hyperparameter training. Set parameters manually for higher accuracy.

param_grid = {  #hyper parameters below
    'n_estimators': [100, 200, 300, 500], #number of trees
    'learning_rate': [0.01, 0.1, 0.2],  #rate at which model changes
    'max_depth': [3, 6, 9], #how deep tree can get
    'min_child_weight': [1, 3, 5],
    'subsample': [0.7, 0.85, 1.0],  #controls no. of observations used to make each tree
    'colsample_bytree': [0.7, 0.85, 1.0],   #how many features used per tree. Smaller -> less complex
    'reg_alpha': [0, 0.01, 0.1, 1, 10, 100],
    'reg_lambda': [0.5, 0.7, 1, 1.3]    #don't set too low
}

xgb_model = xgb.XGBClassifier(random_state = 42)
#cross validation. 10 cross validation folds (short of cv)

#gridsearchCV will take long : every combination of parameters and loops through all of them
#RandomizedSearchCV : Gives faster process

grid_search = RandomizedSearchCV(xgb_model, param_grid, cv=10, scoring="accuracy", n_iter=100, n_jobs=-1, verbose=2, random_state=42) #n_iter : how many combos to search
grid_search.fit(X_train, y_train)

best_xgb = grid_search.best_estimator_      #return with best parameters. gives higher accuracy score

# Best parameters from tuning
print("Best Parameters:", grid_search.best_params_)
print("Best Accuracy:", grid_search.best_score_)


In [ ]:
#Testing Now to X_Test
y_pred = best_xgb.predict(X_test)

test_accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {test_accuracy:.4f}")

# Print classification report
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# Confusion Matrix
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

In [ ]:
#Shows how important each feature is

feature_importances = best_xgb.feature_importances_
feature_names = X.columns

# Sort and plot
sorted_indices = np.argsort(feature_importances)[::-1]
plt.figure(figsize=(10, 5))
plt.bar(range(len(feature_importances)), feature_importances[sorted_indices], align="center")
plt.xticks(range(len(feature_importances)), np.array(feature_names)[sorted_indices], rotation=90)
plt.xlabel("Feature Importance")
plt.title("XGB Feature Importance for Spotify Tracks")
plt.show()